# Projekt: Credit Card Fraud Detection (Detekcja Anomalii)
    
Dataset: `https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud/data`

# 1. Przygotowanie datasetu - import bibliotek + załadowanie dataset'u

In [ ]:
# Podstawowe biblioteki
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Scikit-learn - preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Scikit-learn - modele
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix

import kagglehub


path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Ścieżka do pobranych plików:", path)

# Kagglehub pobiera pliki do dziwnych folderów w cache.
# Musimy znaleźć plik CSV w tym folderze:
csv_file = os.path.join(path, "creditcard.csv")

# Wczytanie
df = pd.read_csv(csv_file)

print(df.info())

print("Pomyślnie załadowano dataset!")

# 2. Pierwsze wizualizacjie (plotting)

### Cel: zwizualizować, że w wybranym datasecie transakcje oszustwa to 0.2%

In [ ]:

colors = ["#6D52E7", "#FF0000"]

plt.pie(
    df['Class'].value_counts(),
    shadow = False,
    colors = colors,
    startangle=90,
    autopct='%1.1f%%',
    labels = ['Normal', 'Fraud']
)
plt.axis('equal')
plt.tight_layout()
plt.show()


## Wniosek: dataset jest ekstremalnie niezbalansowany

### 2.1 Heatmap korelacji (plotting)

In [ ]:
plt.figure(figsize=(30, 24))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

print(df.corr()['Class'].sort_values(ascending=False))


## Z Class najbardziej korelują cechy:
    1. V17 -0.33
    2. V14 -0.30
    3. V12 -0.26

# 3. Preprocessing danych

### Dataset z kaggle ma już większość danych po preprocessingu, więc pozostało dostosować kolumnę `Amount` oraz `Time`

In [ ]:
from sklearn.preprocessing import StandardScaler, RobustScaler

standardScaler = StandardScaler()
robustScaler = RobustScaler()

reshaped_amount = df['Amount'].values.reshape(-1, 1)
reshaped_time = df['Time'].values.reshape(-1, 1)


df['scl_amount'] = robustScaler.fit_transform(reshaped_amount)
df['scl_time'] = robustScaler.fit_transform(reshaped_time)

df.drop(['Time', 'Amount'], axis=1, inplace=True)

print(df.columns)

# 4. Podzial na zbior treningowy i testowy
Przygotowujemy dane i zachowujemy proporcje klas (stratyfikacja).


In [ ]:
# Przygotowanie danych
X = df.drop('Class', axis=1)
y = df['Class']

# Stratyfikowany podzial
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Rozklad klas (train):')
print(y_train.value_counts(normalize=True))
print('Rozklad klas (test):')
print(y_test.value_counts(normalize=True))


# 5. Trenowanie i ewaluacja modelu
Uzywamy modelu bazowego z uwzglednieniem niezbalansowanych klas.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('Classification report:')
print(classification_report(y_test, y_pred, digits=4))

print('Confusion matrix:')
cm = confusion_matrix(y_test, y_pred)
print(cm)

roc_auc = roc_auc_score(y_test, y_proba)
avg_prec = average_precision_score(y_test, y_proba)
print('ROC AUC:', round(roc_auc, 4))
print('Average precision (PR AUC):', round(avg_prec, 4))

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title('ROC Curve')
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.title('Precision-Recall Curve')
plt.show()
